### Read Data and Import Necessary Libraries

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../raw_data/sf_data.db")

query = "SELECT * FROM guild_tracking WHERE stat_type = 'Level'"
level_raw_data_df = pd.read_sql_query(query, conn)

conn.close()

### Level Progress

In [4]:
# get values for each week
weeks = sorted(level_raw_data_df['timestamp'].unique())

# first week (CW17_26)
level_progress = level_raw_data_df[level_raw_data_df['timestamp'] == weeks[0]][['player_name', 'value']]
level_progress.columns = ['Player Name', weeks[0]]

# loop to automatically add new weeks
for week in weeks[1:]:
    current_week_df = level_raw_data_df[level_raw_data_df['timestamp'] == week][['player_name', 'value']]
    current_week_df.columns = ['Player Name', week]
    level_progress = pd.merge(level_progress, current_week_df, on='Player Name', how='outer')

# forward fill for players who left and backward fill for players who joined
level_progress[weeks] = level_progress[weeks].ffill(axis=1).bfill(axis=1)

# compute progress
if len(weeks) > 1:
    latest = weeks[-1]
    previous = weeks[-2]
    level_progress['Weekly_Diff'] = level_progress[latest] - level_progress[previous]
    level_progress['Total_Diff'] = level_progress[latest] - level_progress[weeks[0]]

print(level_progress.sort_values(by=weeks[-1], ascending=False))

         Player Name  2026-04-20
1                Tao         530
3              Falke         508
4             Wutbob         507
0            Paladin         495
5             Bluex3         491
6            Liiight         488
7          Sharandra         484
8        Sigurlasius         467
9          Kampfbock         463
10  Russischer Golum         460
2          StupidHoe         456
11            Raguos         453
12           Pauliv4         451
13          Bagarrao         450
14         Bl4ckless         449
15         Wolff0303         447
16             Fasta         447
17           Swagboi         446
18         FrankoSan         445
19  Bumblebee Hummel         443
20          Restless         442
21        Schmollutz         440
22         ShangriLa         438
23            CortaX         437
24          Flosse97         436
25  Nitzodon Oworotz         430
26      HER-WIEDZMIN         428
27             Borán         426
28         Major Tom         426
29       x